<a href="https://colab.research.google.com/github/TRIAXIS-MOF/TRIAXIS-MOF/blob/main/Input_for_DFT%2C_xTB_Conformational_Energy%2C_and_Geometry_Distortion_Bound_within_the_SBU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Catatan: Pada 3 Tools (Input DFT, Energi Konformasi xTB, dan Distorsi Geometri) file terunduh di files colab harus dihapus untuk memulai setiap tools agar tidak double file yang menyebabkan file error

In [6]:
!pip install rdkit
!pip install ase
!pip install py3Dmol

import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import py3Dmol
from ase import Atoms
from ase.io import write
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from ase.io import read
from itertools import combinations
from google.colab import files
from IPython.display import display, HTML

Ubah output BURAI menjadi CIF atau xyz file

In [7]:
atomic_positions_text = """
C      13.149607   6.167738   6.524006
C      12.294671   7.300114   7.001337
C      11.781530   7.399088   8.301048
H      11.969771   6.623498   9.048326
C      10.990181   8.480183   8.672863
H      10.605361   8.539886   9.690702
C      10.710142   9.502610   7.757016
C      11.996400   8.314031   6.080196
H      12.388175   8.225006   5.066640
C      11.231690   9.407699   6.458116
H      11.032786  10.212093   5.749835
C       9.955401  10.714171   8.157579
N      10.492073  11.877401   7.761677
O      13.667354   5.329358   7.473284
O      13.397165   5.970983   5.351697
C       5.638304   6.139744  10.616163
C       6.479055   7.279894  10.125507
C       6.970381   7.394460   8.818454
H       6.749850   6.644217   8.053708
C       7.741969   8.487370   8.440157
H       8.103331   8.563024   7.414806
C       8.032232   9.503523   9.359275
C       6.773876   8.298087  11.044196
H       6.386249   8.203793  12.059024
C       7.529354   9.397852  10.664374
H       7.738933  10.194598  11.378326
C       8.783348  10.714886   8.954065
N       8.245813  11.878518   9.345530
O       5.404624   5.105377   9.752187
O       5.163770   6.100031  11.733264
C      13.118758  17.617343   6.620649
C      12.271438  16.449121   6.967997
C      11.785642  16.354056   8.278763
H      12.043690  17.137970   8.991067
C      10.999025  15.275270   8.657318
H      10.634758  15.208376   9.682819
C      10.703304  14.253978   7.740940
C      11.968415  15.440027   6.045245
H      12.344510  15.510424   5.026022
C      11.205948  14.346887   6.435198
H      10.998976  13.543718   5.728028
C       9.954003  13.042108   8.151127
O      13.468235  18.482918   7.406678
O      13.486635  17.635875   5.302922
C       5.636384  17.612627  10.615570
C       6.478475  16.474076  10.125199
C       6.970804  16.361100   8.818611
H       6.752630  17.112501   8.054535
C       7.744562  15.269948   8.440532
H       8.109034  15.195415   7.416158
C       8.034661  14.253665   9.359437
C       6.774675  15.456591  11.044079
H       6.389479  15.550708  12.059579
C       7.531254  14.358073  10.664430
H       7.742280  13.561675  11.378447
C       8.783714  13.041584   8.952391
O       5.402192  18.647478   9.752300
O       5.159302  17.650500  11.731573
H       5.860291   5.252974   8.899068
H      13.463974   5.656837   8.372437
H       5.864820  18.504842   8.902191
H      14.045510  18.435541   5.189790
"""

# Parsing otomatis
symbols = []
positions = []

for line in atomic_positions_text.strip().splitlines():
    parts = line.split()
    symbols.append(parts[0])
    positions.append([float(x) for x in parts[1:4]])

# Buat object Atoms
molecule = Atoms(symbols=symbols, positions=positions)

# Cek parsing
print("Jumlah atom:", len(molecule))
print("5 atom pertama dan koordinatnya:")
for i, atom in enumerate(molecule[:5]):
    print(f"{atom.symbol} : {atom.position}")

# Simpan ke CIF atau XYZ
write("NAWXER_Linker_Free Hasil Opt.cif", molecule)
write("NAWXER_Linker_Free Hasil Opt.xyz", molecule)

print("✅ CIF dan XYZ berhasil dibuat!")

Jumlah atom: 62
5 atom pertama dan koordinatnya:
C : [13.149607  6.167738  6.524006]
C : [12.294671  7.300114  7.001337]
C : [11.78153   7.399088  8.301048]
H : [11.969771  6.623498  9.048326]
C : [10.990181  8.480183  8.672863]
✅ CIF dan XYZ berhasil dibuat!


OpenBabel dan UFF (hanya atom H saja yang mengalami relaksasi (mobile))

Upload file xyz linker embed random H cap untuk dibaca open babel

In [8]:
!apt-get install openbabel
!obabel "NAWXER_Linker_Embed_Random H Cap.xyz" -O linker.mol

mol = Chem.MolFromMolFile("linker.mol", removeHs=False)

# UFF heavy atom fixed
ff = AllChem.UFFGetMoleculeForceField(mol)
for i, atom in enumerate(mol.GetAtoms()):
    if atom.GetSymbol() != "H":
        ff.AddFixedPoint(i)

ff.Minimize(maxIts=500)

with open("NAWXER_Linker_Embed_After_UFF.xyz", "w") as f:
    f.write(Chem.MolToXYZBlock(mol))

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openbabel is already the newest version (3.1.1+dfsg-6ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
1 molecule converted


Cek RMSD setelah rotasi dan translasi dihilangkan dan yang dihitung hanya perubahan jarak internal heavy atom (ini sebagai syarat/cek untuk molekul input DFT)

In [9]:
mol1 = Chem.MolFromXYZFile("NAWXER_Linker_Embed_Random H Cap.xyz")
mol2 = Chem.MolFromXYZFile("NAWXER_Linker_Embed_After_UFF.xyz")

if mol1 is None or mol2 is None:
    raise RuntimeError("GAGAL membaca XYZ. Pastikan format murni!")

# Heavy atom indices
idx = [i for i,a in enumerate(mol1.GetAtoms()) if a.GetSymbol() != "H"]

# Buat mapping 1-to-1
atom_map = [(i, i) for i in idx]

# Align dan RMSD
rmsd = rdMolAlign.AlignMol(mol2, mol1, atomMap=atom_map)

print("Heavy atom internal RMSD:", rmsd, "Å")

# Evaluasi otomatis
if rmsd < 0.02:
    print("Status: SEMPURNA → embedded benar")
elif rmsd < 0.05:
    print("Status: AMAN → bisa lanjut DFT")
elif rmsd < 0.1:
    print("Status: RAGU → cek visual")
else:
    print("Status: SALAH → struktur embedded rusak")

Heavy atom internal RMSD: 5.097603643654335e-07 Å
Status: SEMPURNA → embedded benar


Perhitungan energi konformasi linker dengan xTB

Download xTB untuk perhitungan energi konformasi linker

In [10]:
%%bash
rm -rf xtb*

wget https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz

# extract
tar -xf xtb-6.7.1-linux-x86_64.tar.xz

# folder hasil extract
DIR=$(tar -tf xtb-6.7.1-linux-x86_64.tar.xz | head -n 1 | cut -d/ -f1)
echo "Extracted folder: $DIR"

cd "$DIR"

# set environment
export PATH=$PWD/bin:$PATH
export XTBPATH=$PWD/share/xtb

# test
bin/xtb --version

Extracted folder: xtb-dist
      -----------------------------------------------------------      
     |                   =====================                   |     
     |                           x T B                           |     
     |                   =====================                   |     
     |                         S. Grimme                         |     
     |          Mulliken Center for Theoretical Chemistry        |     
     |                    University of Bonn                     |     
      -----------------------------------------------------------      

   * xtb version 6.7.1 (edcfbbe) compiled by 'albert@albert-system' on 2024-07-22



--2026-02-11 13:49:50--  https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/211856832/0ca4f59b-4025-40a9-abba-12e99b3c8d3b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-11T14%3A47%3A43Z&rscd=attachment%3B+filename%3Dxtb-6.7.1-linux-x86_64.tar.xz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-11T13%3A47%3A25Z&ske=2026-02-11T14%3A47%3A43Z&sks=b&skv=2018-11-09&sig=VdapAzAFHjcKTwQ88btUodA72b%2FHz80Y%2FLJNydsgEF4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MDgxOTU5MSwibmJmIjoxNzcwODE3NzkxLCJwYXRoIjoicmVsZWF

Menambahkan xTB ke path

In [11]:
import os

os.environ["PATH"] += ":" + "/content/xtb-dist/bin"
os.environ["XTBPATH"] = "/content/xtb-dist/share/xtb"

Upload file linker embedd after UFF

In [12]:
from google.colab import files
files.upload()

Saving FATQID_Linker_Embed_After_UFF.xyz to FATQID_Linker_Embed_After_UFF.xyz


{'FATQID_Linker_Embed_After_UFF.xyz': b'37\nProperties=species:S:1:pos:R:3 pbc="F F F"\nC      8.773400    9.153000   17.322200\nC      9.516900    9.793900   16.205700\nC      9.104500   11.014600   15.701100\nH      8.233944   11.514115   16.108198\nC      9.773000   11.623700   14.652200\nH      9.386110   12.555952   14.261339\nC     11.306700    9.821400   14.595800\nH     12.190288    9.324225   14.216290\nC     10.616400    9.224800   15.667200\nH     10.992618    8.280965   16.042180\nC     10.894400   10.995900   14.082700\nC     11.709600   13.023100   12.876000\nH     11.296438   13.640089   13.662287\nC     11.597200   11.632800   12.946500\nC     12.206300   10.880300   11.937200\nH     12.162482    9.800632   11.977260\nC     12.893500   11.517500   10.888300\nC     12.918500   12.905700   10.913700\nH     13.406921   13.443164   10.112175\nC     13.596300   10.758800    9.839500\nC     13.096500    9.574700    9.419400\nH     12.164174    9.189073    9.811203\nC     13.7

Perhitungan energi linker embedd dengan xTB

In [13]:
%%bash
xtb FATQID_Linker_Embed_After_UFF.xyz --gfn2 | tee xtb.out

      -----------------------------------------------------------      
     |                   =====================                   |     
     |                           x T B                           |     
     |                   =====================                   |     
     |                         S. Grimme                         |     
     |          Mulliken Center for Theoretical Chemistry        |     
     |                    University of Bonn                     |     
      -----------------------------------------------------------      

   * xtb version 6.7.1 (edcfbbe) compiled by 'albert@albert-system' on 2024-07-22

   xtb is free software: you can redistribute it and/or modify it under
   the terms of the GNU Lesser General Public License as published by
   the Free Software Foundation, either version 3 of the License, or
   (at your option) any later version.
   
   xtb is distributed in the hope that it will be useful,
   but WITHOUT ANY WARRANTY;

normal termination of xtb


Perhitungan energi free (linker embedd setelah optimasi geometri) dengan xTB

In [14]:
!xtb FATQID_Linker_Embed_After_UFF.xyz --opt --gfn2 | tee xtb_opt.out

      -----------------------------------------------------------      
     |                   =====================                   |     
     |                           x T B                           |     
     |                   =====================                   |     
     |                         S. Grimme                         |     
     |          Mulliken Center for Theoretical Chemistry        |     
     |                    University of Bonn                     |     
      -----------------------------------------------------------      

   * xtb version 6.7.1 (edcfbbe) compiled by 'albert@albert-system' on 2024-07-22

   xtb is free software: you can redistribute it and/or modify it under
   the terms of the GNU Lesser General Public License as published by
   the Free Software Foundation, either version 3 of the License, or
   (at your option) any later version.
   
   xtb is distributed in the hope that it will be useful,
   but WITHOUT ANY WARRANTY;

Energi linker free dan embedded hasil perhitungan xTB

In [15]:
import re

# Energi linker embedded
with open("xtb.out") as f:
    txt = f.read()

E = float(re.search(r"TOTAL ENERGY\s+(-?\d+\.\d+)", txt).group(1))

print("Single Point Energy")
print("--------------------")
print("Hartree :", E)
print("Rydberg :", E*2)
print("kcal/mol:", E*627.509)

# Energi linker free (hasil optimasi)
with open("xtb_opt.out") as f:
    txt = f.read()

Eopt = float(re.findall(r"TOTAL ENERGY\s+(-?\d+\.\d+)", txt)[-1])

print("Optimized Energy")
print("----------------")
print("Hartree :", Eopt)
print("Rydberg :", Eopt*2)
print("kcal/mol:", Eopt*627.509)

Single Point Energy
--------------------
Hartree : -66.457313707197
Rydberg : -132.914627414394
kcal/mol: -41702.56246708948
Optimized Energy
----------------
Hartree : -66.501664690907
Rydberg : -133.003329381814
kcal/mol: -41730.39310852637


Perhitungan energi konformasi banyak linker dengan xTB

Upload file linker embedded after UFF untuk kemudian dihitung nilai energi dasarnya dan energi setelah optimasi geometri dengan xTB

In [4]:
from google.colab import files
files.upload()   # upload banyak file xyz

Saving FATQID_Linker_Embed_After_UFF.xyz to FATQID_Linker_Embed_After_UFF.xyz
Saving NAWXER_Linker_Embed_After_UFF.xyz to NAWXER_Linker_Embed_After_UFF.xyz
Saving VOLPET_Linker_Embed_After_UFF.xyz to VOLPET_Linker_Embed_After_UFF.xyz


{'FATQID_Linker_Embed_After_UFF.xyz': b'37\nProperties=species:S:1:pos:R:3 pbc="F F F"\nC      8.773400    9.153000   17.322200\nC      9.516900    9.793900   16.205700\nC      9.104500   11.014600   15.701100\nH      8.233944   11.514115   16.108198\nC      9.773000   11.623700   14.652200\nH      9.386110   12.555952   14.261339\nC     11.306700    9.821400   14.595800\nH     12.190288    9.324225   14.216290\nC     10.616400    9.224800   15.667200\nH     10.992618    8.280965   16.042180\nC     10.894400   10.995900   14.082700\nC     11.709600   13.023100   12.876000\nH     11.296438   13.640089   13.662287\nC     11.597200   11.632800   12.946500\nC     12.206300   10.880300   11.937200\nH     12.162482    9.800632   11.977260\nC     12.893500   11.517500   10.888300\nC     12.918500   12.905700   10.913700\nH     13.406921   13.443164   10.112175\nC     13.596300   10.758800    9.839500\nC     13.096500    9.574700    9.419400\nH     12.164174    9.189073    9.811203\nC     13.7

Perhitungan energi konformasi linker

In [5]:
import glob
import subprocess
import re
import pandas as pd

results = []

for xyz in glob.glob("*.xyz"):
    name = xyz.replace(".xyz","")

    print(f"Processing {xyz} ...")

    # single point
    sp_cmd = f"xtb {xyz} --gfn2 | tee {name}_sp.out"
    subprocess.run(sp_cmd, shell=True)

    # optimization
    opt_cmd = f"xtb {xyz} --opt --gfn2 | tee {name}_opt.out"
    subprocess.run(opt_cmd, shell=True)

    # parse energies
    Esp = float(re.search(
        r"TOTAL ENERGY\s+(-?\d+\.\d+)",
        open(f"{name}_sp.out").read()
    ).group(1))

    Eopt = float(re.findall(
        r"TOTAL ENERGY\s+(-?\d+\.\d+)",
        open(f"{name}_opt.out").read()
    )[-1])

    dE = Esp - Eopt

    results.append([name, Esp, Eopt, dE])

# Hasil
df = pd.DataFrame(results, columns=[
    "Molecule",
    "Single Point (Eh)",
    "Optimized (Eh)",
    "Conformation ΔE (Eh)"
])

df["Single (kcal/mol)"] = df["Single Point (Eh)"] * 627.509
df["Optimized (kcal/mol)"] = df["Optimized (Eh)"] * 627.509
df["ΔE (kcal/mol)"] = df["Conformation ΔE (Eh)"] * 627.509

df["Single (Ry)"] = df["Single Point (Eh)"] * 2
df["Optimized (Ry)"] = df["Optimized (Eh)"] * 2
df["ΔE (Ry)"] = df["Conformation ΔE (Eh)"] * 2

df

Processing FATQID_Linker_Embed_After_UFF.xyz ...
Processing NAWXER_Linker_Embed_After_UFF.xyz ...
Processing VOLPET_Linker_Embed_After_UFF.xyz ...


,Molecule,Single Point (Eh),Optimized (Eh),Conformation ΔE (Eh),Single (kcal/mol),Optimized (kcal/mol),ΔE (kcal/mol),Single (Ry),Optimized (Ry),ΔE (Ry)
0,FATQID_Linker_Embed_After_UFF,-66.457314,-66.501665,0.044351,-41702.562467,-41730.393109,27.830641,-132.914627,-133.003329,0.088702
1,NAWXER_Linker_Embed_After_UFF,-117.030478,-117.110537,0.080059,-73437.678158,-73487.916127,50.237969,-234.060956,-234.221075,0.160119
2,VOLPET_Linker_Embed_After_UFF,-34.360648,-34.394272,0.033624,-21561.616010,-21582.715468,21.099458,-68.721296,-68.788545,0.067248


Perhitungan RMSD, delta length dan angle pada struktur embedded dan free

In [16]:
# Fungsi RMSD dengan Kabsch alignment
def kabsch_rmsd(P, Q):
    """
    P, Q: numpy array (N,3)
    Returns RMSD after optimal alignment of P onto Q
    """
    # Center the structures
    P_centered = P - P.mean(axis=0)
    Q_centered = Q - Q.mean(axis=0)

    # Covariance matrix
    C = np.dot(P_centered.T, Q_centered)

    # SVD
    V, S, Wt = np.linalg.svd(C)
    d = np.sign(np.linalg.det(np.dot(Wt.T, V.T)))
    D = np.diag([1,1,d])
    U = np.dot(Wt.T, np.dot(D, V.T))

    # Rotate P
    P_rot = np.dot(P_centered, U)

    # RMSD
    rmsd = np.sqrt(np.mean(np.sum((P_rot - Q_centered)**2, axis=1)))
    return rmsd, P_rot

# Jari-jari kovalen (Å)
cov_radii = {
    'H': 0.31, 'C': 0.76, 'N': 0.71, 'O': 0.66, 'F': 0.57,
    'P': 1.07, 'S': 1.05, 'Cl': 1.02, 'Br': 1.20, 'I': 1.39
}

# Deteksi ikatan otomatis
def detect_bonds(symbols, coords, scale=1.2):
    bonds = []
    N = len(symbols)
    for i,j in combinations(range(N),2):
        if symbols[i] not in cov_radii or symbols[j] not in cov_radii:
            continue
        cutoff = scale*(cov_radii[symbols[i]] + cov_radii[symbols[j]])
        dist = np.linalg.norm(coords[i]-coords[j])
        if dist <= cutoff:
            bonds.append((i,j))
    return bonds

# Hitung panjang ikatan & sudut
def bond_lengths(coords, bonds):
    return np.array([np.linalg.norm(coords[i]-coords[j]) for i,j in bonds])

def bond_angles(coords, bonds):
    angles = []
    triplets = []
    neighbor_dict = {}
    for i,j in bonds:
        neighbor_dict.setdefault(i, []).append(j)
        neighbor_dict.setdefault(j, []).append(i)
    for center, neighbors in neighbor_dict.items():
        for i in range(len(neighbors)):
            for j in range(i+1, len(neighbors)):
                a,b = neighbors[i], neighbors[j]
                v1 = coords[a]-coords[center]
                v2 = coords[b]-coords[center]
                cos_theta = np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))
                cos_theta = np.clip(cos_theta, -1.0, 1.0)
                theta = np.arccos(cos_theta)*180/np.pi
                angles.append(theta)
                triplets.append((a,center,b))
    return np.array(angles), triplets

# Fungsi utama evaluasi distorsi
def evaluate_distortion(file_free, file_embedded, scale=1.2):
    atoms_free = read(file_free)
    atoms_emb = read(file_embedded)

    symbols_free = atoms_free.get_chemical_symbols()
    symbols_emb = atoms_emb.get_chemical_symbols()

    coords_free = atoms_free.get_positions()
    coords_emb = atoms_emb.get_positions()
    coords_emb_ordered = coords_emb

    # RMSD
    rmsd, coords_free_aligned = kabsch_rmsd(coords_free, coords_emb_ordered)

    # Deteksi ikatan
    bonds_free = detect_bonds(symbols_free, coords_free, scale=1.5)
    lengths_free = bond_lengths(coords_free, bonds_free)
    lengths_emb = bond_lengths(coords_emb_ordered, bonds_free)

    angles_free, triplets = bond_angles(coords_free, bonds_free)
    angles_emb, _ = bond_angles(coords_emb_ordered, bonds_free)
    delta_lengths = lengths_emb - lengths_free
    delta_angles = angles_emb - angles_free

    df_bonds = pd.DataFrame({
        'atom_i':[i for i,j in bonds_free],
        'atom_j':[j for i,j in bonds_free],
        'length_free':lengths_free,
        'length_embedded':lengths_emb,
        'delta_length':delta_lengths
    })

    df_angles = pd.DataFrame({
        'atom_i':[a for a,c,b in triplets],
        'atom_center':[c for a,c,b in triplets],
        'atom_k':[b for a,c,b in triplets],
        'angle_free':angles_free,
        'angle_embedded':angles_emb,
        'delta_angle':delta_angles
    })

    return rmsd, df_bonds, df_angles, symbols_free, coords_free_aligned, coords_emb_ordered

# Upload CIF
print("Upload CIF file linker free:")
uploaded_free = files.upload()
file_free = list(uploaded_free.keys())[0]

print("Upload CIF file linker embedded:")
uploaded_emb = files.upload()
file_embedded = list(uploaded_emb.keys())[0]

# Evaluasi distorsi
rmsd, df_bonds, df_angles, symbols_free, coords_free_aligned, coords_emb_ordered = evaluate_distortion(file_free, file_embedded)

print(f"\nRMSD (Å): {rmsd:.4f}")
print("\nΔBond Lengths:")
display(df_bonds)
print("\nΔBond Angles:")
display(df_angles)

# Statistik Distorsi
ME_delta_length = df_bonds['delta_length'].mean()
ME_delta_angle  = df_angles['delta_angle'].mean()

print("\n===== Mean Error (ME) =====")
print(f"ME ΔBond Length (Å): {ME_delta_length:.6f}")
print(f"ME ΔBond Angle (deg): {ME_delta_angle:.4f}")

print("\nInterpretation:")
if ME_delta_length > 0:
    print("→ Secara rata-rata terjadi pemanjangan ikatan")
else:
    print("→ Secara rata-rata terjadi pemendekan ikatan")

if ME_delta_angle > 0:
    print("→ Sudut ikatan cenderung melebar")
else:
    print("→ Sudut ikatan cenderung menyempit")

Upload CIF file linker free:


Saving FATQID_Linker_Free Hasil Opt.xyz to FATQID_Linker_Free Hasil Opt.xyz
Upload CIF file linker embedded:


Saving FATQID_Linker_Embed_After_UFF.xyz to FATQID_Linker_Embed_After_UFF (1).xyz

RMSD (Å): 0.0810

ΔBond Lengths:


,atom_i,atom_j,length_free,length_embedded,delta_length
0,0,1,1.482244,1.486646,0.004402
1,0,33,1.220769,1.247745,0.026976
2,0,34,1.369356,1.257253,-0.112103
3,1,2,1.400580,1.383764,-0.016816
4,1,8,1.400696,1.350095,-0.050600
5,2,3,1.090285,1.083103,-0.007182
6,2,4,1.387361,1.384950,-0.002411
7,4,5,1.090880,1.082382,-0.008498
8,4,10,1.406416,1.405703,-0.000713
9,6,7,1.090926,1.082561,-0.008365



ΔBond Angles:


,atom_i,atom_center,atom_k,angle_free,angle_embedded,delta_angle
0,1,0,33,125.104174,118.759682,-6.344493
1,1,0,34,112.723650,115.736216,3.012567
2,33,0,34,122.170975,125.482155,3.311179
3,0,1,2,118.421028,120.339203,1.918176
4,0,1,8,122.312437,121.676372,-0.636065
5,2,1,8,119.263409,117.976867,-1.286542
6,0,34,35,106.000625,121.083968,15.083343
7,1,2,3,118.721696,120.618799,1.897103
8,1,2,4,120.364727,121.351958,0.987230
9,3,2,4,120.911086,118.021181,-2.889905



===== Mean Error (ME) =====
ME ΔBond Length (Å): -0.011986
ME ΔBond Angle (deg): 0.4950

Interpretation:
→ Secara rata-rata terjadi pemendekan ikatan
→ Sudut ikatan cenderung melebar


Visualiasi distori setelah free linker digeser agar tidak overlapping tapi perhitungan tetap di 1 pusat yang sama

In [17]:
# Safety check
assert len(symbols_free) == coords_free_aligned.shape[0] == coords_emb_ordered.shape[0], \
    "ERROR: Jumlah atom tidak konsisten!"

# Centering
coords_free_centered = coords_free_aligned - coords_free_aligned.mean(axis=0)
coords_emb_centered  = coords_emb_ordered  - coords_emb_ordered.mean(axis=0)

# Atom-wise displacement (Δr_i)
delta_r = np.linalg.norm(coords_emb_centered - coords_free_centered, axis=1)

df_atoms = pd.DataFrame({
    'atom_index': np.arange(len(symbols_free)),
    'element': symbols_free,
    'delta_r (Å)': delta_r
})

print("\nAtom Δr minimum:")
display(df_atoms.loc[df_atoms['delta_r (Å)'].idxmin()])

print("\nAtom Δr maksimum:")
display(df_atoms.loc[df_atoms['delta_r (Å)'].idxmax()])

print("\nAtom-wise displacement (Δr_i):")
display(df_atoms)

print(f"\nΔr min = {delta_r.min():.3f} Å")
print(f"Δr max = {delta_r.max():.3f} Å")

# Normalisasi warna
delta_min = delta_r.min()
delta_max = delta_r.max()

norm = mcolors.Normalize(vmin=delta_min, vmax=delta_max)
cmap = cm.get_cmap('viridis')

def displacement_to_color(val, vmin, vmax):
    t = norm(val)
    r, g, b, _ = cmap(t)
    return f"rgb({int(255*r)},{int(255*g)},{int(255*b)})"

colors = [displacement_to_color(v, delta_min, delta_max) for v in delta_r]
assert None not in colors, "Ada atom tanpa warna!"

# Visual offset (hanya untuk visualisasi)
VISUAL_SHIFT = np.array([0.0, 4.9, 0.0])  # geser embedded ke (+y)
coords_emb_visual = coords_emb_centered + VISUAL_SHIFT

# py3Dmol visualization: Overlay + RMSD mapping
view = py3Dmol.view(width=900, height=650)

# Free linker (aligned + centered)
xyz_free = f"{len(symbols_free)}\nfree_aligned_centered\n"
for s, c in zip(symbols_free, coords_free_centered):
    xyz_free += f"{s} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n"

view.addModel(xyz_free, "xyz")
view.setStyle(
    {'model': 0},
    {
        'stick': {'radius': 0.13, 'color': 'lightgray'},
        'sphere': {'scale': 0.18, 'color': 'lightgray'}
    }
)

# Embedded linker
xyz_emb = f"{len(symbols_free)}\nembedded_centered_visual\n"
for s, c in zip(symbols_free, coords_emb_visual):
    xyz_emb += f"{s} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n"

view.addModel(xyz_emb, "xyz")
view.setStyle({'model': 1}, {})

for i, color in enumerate(colors):
    view.setStyle(
        {'model': 1, 'index': i},
        {
            'sphere': {'scale': 0.32, 'color': color},
            'stick':  {'radius': 0.18, 'color': color}
        }
    )

# View settings
view.zoomTo()
view.setBackgroundColor('white')

view.setViewStyle({
    'style': 'outline',
    'color': 'black',
    'width': 0.02
})

# Axis legend (ON/OFF)
SHOW_AXIS = True

if SHOW_AXIS:
    axis_len = 2.5

    label_style = {
        'backgroundColor': 'rgba(255,255,255,0.85)',
        'fontSize': 18,
        'padding': 4,
        'borderRadius': 4,
        'fontColor': 'black'
    }

    # X-axis
    view.addArrow({
        'start': {'x': 0, 'y': 0, 'z': 0},
        'end':   {'x': axis_len, 'y': 0, 'z': 0},
        'radius': 0.08,
        'color': 'red'
    })
    #view.addLabel("X", {
        #**label_style,
        #'position': {'x': axis_len + 0.3, 'y': 0, 'z': 0},
        #'fontColor': 'red'
    #})

    # Y-axis
    view.addArrow({
        'start': {'x': 0, 'y': 0, 'z': 0},
        'end':   {'x': 0, 'y': axis_len, 'z': 0},
        'radius': 0.08,
        'color': 'green'
    })
    #view.addLabel("Y", {
        #**label_style,
        #'position': {'x': 0, 'y': axis_len + 0.3, 'z': 0},
        #'fontColor': 'green'
    #})

    # Z-axis
    view.addArrow({
        'start': {'x': 0, 'y': 0, 'z': 0},
        'end':   {'x': 0, 'y': 0, 'z': axis_len},
        'radius': 0.08,
        'color': 'blue'
    })
    #view.addLabel("Z", {
        #**label_style,
        #'position': {'x': 0, 'y': 0, 'z': axis_len + 0.3},
        #'fontColor': 'blue'
    #})

view.show()

# Fixed color legend (toggleable)
levels = np.linspace(delta_min, delta_max, 5)
legend_colors = [displacement_to_color(v, delta_min, delta_max) for v in levels]

legend_items = ""
for v, col in zip(levels, legend_colors):
    legend_items += f"""
    <div style="display:flex; align-items:center; margin-bottom:4px;">
        <div style="
            width:18px;
            height:18px;
            background:{col};
            margin-right:8px;
            border-radius:3px;
        "></div>
        <span style="font-size:13px;">Δr = {v:.2f} Å</span>
    </div>
    """

display(HTML(f"""
<div id="legend-box" style="
    position: fixed;
    bottom: 20px;
    left: 20px;
    background: rgba(255,255,255,0.9);
    padding: 10px 12px;
    border-radius: 8px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.15);
    font-family: Arial, sans-serif;
    z-index: 9999;
">
    <b style="font-size:14px;">Atom-wise Δr</b>
    <div style="margin-top:8px;">
        {legend_items}
    </div>
</div>
"""))

display(HTML("""
<button onclick="
    const el = document.getElementById('legend-box');
    el.style.display = (el.style.display === 'none') ? 'block' : 'none';
" style="
    position: fixed;
    bottom: 20px;
    left: 200px;
    z-index: 10001;
    padding: 6px 10px;
    font-size: 13px;
    background: white;
    border: 1px solid #aaa;
    border-radius: 6px;
    cursor: pointer;
">
Legend
</button>
"""))


Atom Δr minimum:


,15
atom_index,15
element,H
delta_r (Å),0.012597



Atom Δr maksimum:


,30
atom_index,30
element,O
delta_r (Å),0.249346



Atom-wise displacement (Δr_i):


,atom_index,element,delta_r (Å)
0,0,C,0.033222
1,1,C,0.013592
2,2,C,0.028752
3,3,H,0.063316
4,4,C,0.023630
5,5,H,0.015522
6,6,C,0.040895
7,7,H,0.030345
8,8,C,0.054826
9,9,H,0.055024



Δr min = 0.013 Å
Δr max = 0.249 Å


/tmp/ipython-input-2038251782.py:35: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('viridis')


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Visualisasi distori overlapping di 1 pusat

In [18]:
# Safety check
assert len(symbols_free) == coords_free_aligned.shape[0] == coords_emb_ordered.shape[0], \
    "ERROR: Jumlah atom tidak konsisten!"

# Centering
coords_free_centered = coords_free_aligned - coords_free_aligned.mean(axis=0)
coords_emb_centered  = coords_emb_ordered  - coords_emb_ordered.mean(axis=0)

# Atom-wise displacement (Δr_i)
delta_r = np.linalg.norm(coords_emb_centered - coords_free_centered, axis=1)

df_atoms = pd.DataFrame({
    'atom_index': np.arange(len(symbols_free)),
    'element': symbols_free,
    'delta_r (Å)': delta_r
})

print("\nAtom Δr minimum:")
display(df_atoms.loc[df_atoms['delta_r (Å)'].idxmin()])

print("\nAtom Δr maksimum:")
display(df_atoms.loc[df_atoms['delta_r (Å)'].idxmax()])

print("\nAtom-wise displacement (Δr_i):")
display(df_atoms)

print(f"\nΔr min = {delta_r.min():.3f} Å")
print(f"Δr max = {delta_r.max():.3f} Å")

# Normalisasi warna
delta_min = delta_r.min()
delta_max = delta_r.max()

norm = mcolors.Normalize(vmin=delta_min, vmax=delta_max)
cmap = cm.get_cmap('viridis')

def displacement_to_color(val, vmin, vmax):
    t = norm(val)
    r, g, b, _ = cmap(t)
    return f"rgb({int(255*r)},{int(255*g)},{int(255*b)})"

colors = [displacement_to_color(v, delta_min, delta_max) for v in delta_r]
assert None not in colors, "Ada atom tanpa warna!"

# py3Dmol visualization: Overlay + RMSD mapping
view = py3Dmol.view(width=900, height=650)

# Free linker (aligned + centered)
xyz_free = f"{len(symbols_free)}\nfree_aligned_centered\n"
for s, c in zip(symbols_free, coords_free_centered):
    xyz_free += f"{s} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n"

view.addModel(xyz_free, "xyz")
view.setStyle(
    {'model': 0},
    {
        'stick': {'radius': 0.13, 'color': 'lightgray'},
        'sphere': {'scale': 0.18, 'color': 'lightgray'}
    }
)

# Embedded linker
xyz_emb = f"{len(symbols_free)}\nembedded_centered\n"
for s, c in zip(symbols_free, coords_emb_centered):
    xyz_emb += f"{s} {c[0]:.6f} {c[1]:.6f} {c[2]:.6f}\n"

view.addModel(xyz_emb, "xyz")

view.setStyle({'model': 1}, {})

for i, color in enumerate(colors):
    view.setStyle(
        {'model': 1, 'index': i},
        {
            'sphere': {'scale': 0.32, 'color': color},
            'stick':  {'radius': 0.18, 'color': color}
        }
    )

# View settings
view.zoomTo()
view.setBackgroundColor('white')

view.setViewStyle({
    'style': 'outline',
    'color': 'black',
    'width': 0.02
})

# Legend axis
SHOW_AXIS = False
if SHOW_AXIS:
    axis_len = 2.5

    label_style = {
        'backgroundColor': 'rgba(255,255,255,0.85)',
        'fontSize': 18,
        'padding': 4,
        'borderRadius': 4,
        'fontColor': 'black'
    }

    # X-axis
    view.addArrow({
        'start': {'x': 0, 'y': 0, 'z': 0},
        'end':   {'x': axis_len, 'y': 0, 'z': 0},
        'radius': 0.08,
        'color': 'red'
    })
    view.addLabel("X", {
        **label_style,
        'position': {'x': axis_len + 0.3, 'y': 0, 'z': 0},
        'fontColor': 'red'
    })

    # Y-axis
    view.addArrow({
        'start': {'x': 0, 'y': 0, 'z': 0},
        'end':   {'x': 0, 'y': axis_len, 'z': 0},
        'radius': 0.08,
        'color': 'green'
    })
    view.addLabel("Y", {
        **label_style,
        'position': {'x': 0, 'y': axis_len + 0.3, 'z': 0},
        'fontColor': 'green'
    })

    # Z-axis
    view.addArrow({
        'start': {'x': 0, 'y': 0, 'z': 0},
        'end':   {'x': 0, 'y': 0, 'z': axis_len},
        'radius': 0.08,
        'color': 'blue'
    })
    view.addLabel("Z", {
        **label_style,
        'position': {'x': 0, 'y': 0, 'z': axis_len + 0.3},
        'fontColor': 'blue'
    })


view.show()

# Legend
levels = np.linspace(delta_min, delta_max, 5)
legend_colors = [displacement_to_color(v, delta_min, delta_max) for v in levels]

legend_items = ""
for v, col in zip(levels, legend_colors):
    legend_items += f"""
    <div style="display:flex; align-items:center; margin-bottom:4px;">
        <div style="
            width:18px;
            height:18px;
            background:{col};
            margin-right:8px;
            border-radius:3px;
        "></div>
        <span style="font-size:13px;">Δr = {v:.2f} Å</span>
    </div>
    """

display(HTML(f"""
<div id="legend-box" style="
    position: fixed;
    bottom: 20px;
    left: 20px;
    background: rgba(255,255,255,0.9);
    padding: 10px 12px;
    border-radius: 8px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.15);
    font-family: Arial, sans-serif;
    z-index: 9999;
">
    <b style="font-size:14px;">Atom-wise Δr</b>
    <div style="margin-top:8px;">
        {legend_items}
    </div>
</div>
"""))

display(HTML("""
<button onclick="
    const el = document.getElementById('legend-box');
    el.style.display = (el.style.display === 'none') ? 'block' : 'none';
" style="
    position: fixed;
    bottom: 20px;
    left: 200px;
    z-index: 10001;
    padding: 6px 10px;
    font-size: 13px;
    background: white;
    border: 1px solid #aaa;
    border-radius: 6px;
    cursor: pointer;
">
Legend
</button>
"""))


Atom Δr minimum:


,15
atom_index,15
element,H
delta_r (Å),0.012597



Atom Δr maksimum:


,30
atom_index,30
element,O
delta_r (Å),0.249346



Atom-wise displacement (Δr_i):


,atom_index,element,delta_r (Å)
0,0,C,0.033222
1,1,C,0.013592
2,2,C,0.028752
3,3,H,0.063316
4,4,C,0.023630
5,5,H,0.015522
6,6,C,0.040895
7,7,H,0.030345
8,8,C,0.054826
9,9,H,0.055024



Δr min = 0.013 Å
Δr max = 0.249 Å


/tmp/ipython-input-2315424417.py:35: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('viridis')


3Dmol.js failed to load for some reason. Please check your browser console for error messages.